#  Causal Primary Econometric Analysis

This notebook is for a causal analysis. It includes explicit ambition, identification, specification, results reporting, interpretation, and threats to validity.

## 1. Analysis Ambition

This notebook reports the causal ambition of the primary econometric analysis. The research question asks whether increases in female secondary school enrolment affect fertility rates in a panel of Sub-Saharan African countries.

The analysis is explicitly causal: it attempts to estimate the within-country effect of female secondary enrolment on fertility while controlling for country-specific fixed factors and common year shocks. The goal is not only to describe an association, but to assess whether changes in enrolment are linked to changes in fertility rates after accounting for unobserved heterogeneity across countries and years.

The primary estimated effect is the coefficient on `Female_Secondary_Enrollment_Rate` in a two-way fixed effects regression.

## 2. Data and Sample

- Outcome variable: `Fertility_Rate` (total fertility rate, births per woman).
- Main exposure variable: `Female_Secondary_Enrollment_Rate` (female secondary school enrolment rate, percent of eligible population).
- Panel controls: country fixed effects and year fixed effects are used instead of additional observed covariates.
- Sample: country-year observations from Malawi, Rwanda, Burkina Faso, and Mali.
- Data source: cleaned World Bank indicator panel in `../Data.clean/panel_fixed_effects_data.csv`.

The analysis sample includes 70 country-year observations after dropping missing values for the two key variables. The panel covers years 2001 through 2023 for the included countries, with a balanced regional focus on four Sub-Saharan African economies.

## 3. Identification Strategy and Assumption

The identifying assumption is that, after controlling for country fixed effects and year fixed effects, remaining variation in `Female_Secondary_Enrollment_Rate` is as good as random with respect to `Fertility_Rate`.

This requires:
- no omitted time-varying confounder that affects both changes in female secondary enrolment and changes in fertility within countries,
- no reverse causality where fertility changes directly drive female secondary enrolment within the sample window,
- and stable measurement of the key indicators across countries and years.

The strategy relies on two-way fixed effects because it removes unobserved time-invariant differences across countries and common shocks in each year. The causal interpretation therefore rests on the assumption that any remaining bias is driven by time-varying factors that are not captured by the fixed effects.


In [3]:
import pandas as pd
import statsmodels.formula.api as smf

panel = pd.read_csv('../Data.clean/panel_fixed_effects_data.csv')
panel = panel.dropna(subset=['Female_Secondary_Enrollment_Rate', 'Fertility_Rate'])

print('Number of observations:', len(panel))
print('Country counts:')
print(panel['Country_Name'].value_counts())
print('Year range:', panel['Year'].min(), 'to', panel['Year'].max())
print('\nVariable summary:')
print(panel[['Female_Secondary_Enrollment_Rate', 'Fertility_Rate']].describe())

formula = 'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)'
model = smf.ols(formula, data=panel).fit(
    cov_type='cluster',
    cov_kwds={'groups': panel['Country_Name']}
)

print('\nRegression summary:')
print(model.summary())

coef = model.params['Female_Secondary_Enrollment_Rate']
se = model.bse['Female_Secondary_Enrollment_Rate']
pval = model.pvalues['Female_Secondary_Enrollment_Rate']
print(f"\nEstimated effect: {coef:.4f} (SE = {se:.4f}, p = {pval:.3f})")

Number of observations: 70
Country counts:
Country_Name
Malawi          19
Rwanda          19
Burkina Faso    16
Mali            16
Name: count, dtype: int64
Year range: 2001 to 2023

Variable summary:
       Female_Secondary_Enrollment_Rate  Fertility_Rate
count                         70.000000       70.000000
mean                          30.534741        5.211057
std                           10.329936        0.995512
min                           10.954960        3.648000
25%                           22.896346        4.189750
50%                           33.038252        5.333000
75%                           37.542528        6.095000
max                           48.778516        6.783000

Regression summary:
                            OLS Regression Results                            
Dep. Variable:         Fertility_Rate   R-squared:                       0.951
Model:                            OLS   Adj. R-squared:                  0.922
Method:                 Least Square

c:\Users\jethr\Downloads\Data-Evidence-in-Economics-Empirical-Project-main\Data-Evidence-in-Economics-Empirical-Project-main\Data-Evidence-in-Economics-Empirical-Project\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 26, but rank is 3
  warnings.warn('covariance of constraints does not have full '


## 4. Econometric Specification 

**Functional form:** Linear OLS regression with two-way fixed effects.

**Regression equation:**
`Fertility_Rate_it = alpha_i + gamma_t + beta * Female_Secondary_Enrollment_Rate_it + u_it`

where:
- `Fertility_Rate_it` is the total fertility rate for country `i` in year `t`,
- `alpha_i` is a country fixed effect capturing time-invariant country-level differences (e.g., geography, cultural factors),
- `gamma_t` is a year fixed effect capturing common shocks affecting all countries (e.g., global economic trends),
- `Female_Secondary_Enrollment_Rate_it` is the female secondary school enrollment rate, measured as a percent,
- `beta` is the parameter of interest (the within-country, within-year effect of enrollment on fertility),
- `u_it` is the error term.

**Regressors:**
- Main treatment: `Female_Secondary_Enrollment_Rate` (continuous, percent).
- Fixed effects: `C(Country_Name)` with 4 levels (Malawi, Rwanda, Burkina Faso, Mali) and `C(Year)` with 23 levels (2001–2023).
- No additional observed covariates are included; instead, fixed effects absorb time-invariant heterogeneity and common trends.

**Justification for regressors:** Country fixed effects remove unobserved time-invariant differences across countries that may affect both enrollment and fertility (e.g., institutional quality, religious composition). Year fixed effects remove global shocks and secular fertility trends that affect all countries simultaneously, improving causal identification within countries.

**Sample:**
- Panel of 4 Sub-Saharan African countries: Malawi, Rwanda, Burkina Faso, Mali.
- Years 2001–2023 (23 years).
- Total observations: 70 (after dropping missing values for the two key variables).
- Justification: Countries selected based on data availability,regional representation, and economic similarities to each other within Africa. 

**Error structure:**
- Standard errors are clustered by country to allow arbitrary correlation of errors within each country over time.
- Justification: Panel data often exhibits serial correlation within units; clustering corrects standard errors for this dependence and yields valid inference on the treatment coefficient.


# Regression Table: Female Secondary Enrollment and Fertility Rates

**Dep. var: avg. fertility rate**

| | (1) Country + year FE | (2) Country FE only | (3) Year FE only | (4) Malawi-only OLS |
|---|---|---|---|---|
| **Female secondary enrollment** | −0.0015 | −0.0562*** | −0.0572** | −0.1332*** |
| | (0.0231) | (0.0076) | (0.0185) | (0.0177) |
| **Country fixed effects** | Yes | Yes | No | No |
| **Year fixed effects** | Yes | No | Yes | No |
| **N** | 70 | 70 | 70 | 19 |
| **Adj. R²** | 0.951 | 0.867 | 0.594 | 0.720 |

This regression table reports the key coefficient on female secondary enrolment across four specifications. Standard errors are clustered by country for the panel regressions and are reported in the separate column. The main two-way fixed effects model is the preferred specification because it removes both time-invariant country heterogeneity and common year shocks.

Key points from the table:
- The main country + year fixed effects estimate is essentially zero and statistically insignificant. The coefficient of -0.0015 has a large standard error relative to its magnitude, producing a p-value of 0.948. This suggests no reliable within-country association between female secondary enrolment and fertility changes once country and year effects are controlled.
- The country fixed effects only model shows a stronger negative association (-0.0562) with a highly significant p-value. The change in magnitude and significance compared to the main model indicates that common year shocks matter: omitting year effects lets secular trends drive the result.
- The year fixed effects only model also yields a negative and statistically significant coefficient (-0.0572), but it ignores persistent country-level differences. This again highlights that the choice of fixed effects materially affects the estimated relationship.
- The Malawi-only OLS result is the most negative (-0.1332) and statistically significant, but it is based on only 19 observations and therefore is less generalizable than the broader panel specifications.

The R^2 values show that the two-way fixed effects model explains most of the variation in fertility rates (0.951) by absorbing country and year-level variation. The lower R^2 in the year-only model indicates that country-specific intercepts capture a large portion of the systematic differences in fertility across countries.

## 6. Interpretation

Our analysis looked at whether increasing female secondary education leads to lower fertility rates (fewer children per woman) in four African countries: Malawi, Rwanda, Burkina Faso, and Mali. Here's what we found:

**The Effect Size:** When female secondary school enrollment increases by 1 percentage point (for example, from 50% to 51% of girls enrolled), the average fertility rate changes by only -0.0015 children per woman. To put this in perspective: this effect is so tiny that in a population of 1,000 women, you would expect fewer than 2 fewer children born. In real-world terms, this is almost no change at all.

**Statistical Significance:** The p-value of 0.948 tells us that if there truly were no relationship between female education and fertility in these countries, we would see results like ours 94.8% of the time just by chance. In other words, the pattern we observed is extremely likely to be random noise rather than evidence of a real effect. 

Think of it this way: if you flipped a coin and got heads 10 times in a row, you might think the coin is biased. But actually, that happens randomly about 1 in 1,000 times. A p-value of 0.948 is saying "this pattern is even more likely to be random than a fair coin."

### What This Means Practically

**This is NOT a finding that female education doesn't matter.** Instead, our finding is much more modest: in our specific four countries, during 2001-2023, we cannot detect a consistent, reliable link between changes in female school enrollment and changes in fertility rates within countries when we control for differences between countries and year-to-year global trends.

This could mean several things:
- There might be a real effect that's just too small or takes too long to show up for us to detect with this data,
- Other factors (economic development, healthcare access, cultural changes) might be more important drivers of fertility in these countries, such as the HIV outbreak in Malawi leading to constant drops in fertility rates.
- The relationship may be more complex than a simple linear relationship such as the rise in education in female leads to rise in wages and jobs for women alongside the knowledge from schooling leading to the dropping fertility rates.
- Our data may be too limited (only 4 countries, 23 years) to pick up the true effect.

**Why we can still be confident in our result:** Even though the effect is small and not statistically significant, the structure of our study is strong. We controlled for permanent differences between countries (e.g., geography, culture) and for global shocks that affect all countries the same way (e.g., international economic trends). This helps isolate what's happening within each country over time, making our estimate more reliable as a within-country relationship.

## 7. Threats to Validity

Key threats for the causal interpretation include:
- time-varying omitted variables. Examples include changes in economic growth, health spending, family planning programs, or schooling policy reforms that happen at the same time as female enrollment changes and also affect fertility.
- reverse causality. If fertility changes influence school enrollment (for example, lower fertility makes it easier for girls to stay in school), then the direction of cause is unclear.
- measurement error in the World Bank series. Enrollment and fertility data may be reported with error, and such errors can bias the estimated coefficient toward zero or make standard errors unreliable.
- a small, unbalanced panel. With only four countries and about 70 observations, the analysis has limited statistical power and may be sensitive to one or two unusual country-year observations.
- limited external validity. Results from these four countries may not generalize to other regions or larger samples.
- potential multicollinearity between fixed effects and the treatment variable. With a short panel and strong year and country patterns, it can be hard to separate the effect of enrollment from other time trends.
- missing data decisions. Dropping observations with missing values can change the sample composition, and missing years or indicators may reflect periods of political instability or reporting gaps.

How this analysis reduces those threats:
- country fixed effects absorb all time-invariant differences between countries, such as long-run cultural norms, geography, and baseline institutional quality.
- year fixed effects absorb common shocks that affect all countries at the same time, such as global economic cycles or international policy shifts.
- clustered standard errors allow for arbitrary correlation within countries over time, which is important in panel data with repeated observations for the same country.
- careful data cleaning and consistent file naming ensure the same source data are used across scripts and the notebook.

Still, the most plausible remaining concern is time-varying country-specific confounding. For example, if a country expands health services to teenage girls at the same time as female enrollment improves, and those services directly reduce fertility, then the estimated effect of enrollment may be biased.

### Additional checks that would strengthen confidence
- run the model with alternative control variables if they become available (GDP per capita, health spending, female labor force participation);
- estimate lagged effects of enrollment on fertility, since education may affect fertility with a delay;
- compare results across alternative country groups or periods to test whether the finding is sample-specific;
- inspect residuals and leverage points to determine whether a small number of observations are driving the result;
- use placebo or falsification tests, such as predicting a variable that should not respond to enrollment, to see whether the model is capturing spurious relationships.
